MV Overhead Line Poles - Seriemas
=
This notebook chooses the block of each pole of the Seriemas medium voltage overhead line (RMT) and writes an AutoCAD script that inserts it. The block name is the pole height plus `Simples` (single circuit) or `Duplo` (double circuit), e.g. `14-Duplo`.

The cable polylines come from the DWG. The poles come from an Excel data extraction (`posts.xlsx`) of the same drawing, with their attributes and angles.

**Workflow overview:**
1. Convert the DWG to DXF and extract every model space entity (without DuckDB).
2. Load the pole table from `posts.xlsx` and calculate the rotation angle.
3. Keep the cable polylines.
4. Take the pole height from the `H/CAP.(PLANTA)` attribute.
5. Find the cables that start or end at each pole.
6. Choose `Simples` or `Duplo` from the layers of those cables.
7. Write the `-insert` script.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** the drawing is `all.dwg` in `C:\Users\Usuario\Desktop\seriemas\RMT\all`. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** only entities placed directly in model space are read.
- **Cables:**
  - Every `LWPOLYLINE` in model space is treated as a cable (there's no layer filter). Any other polyline that starts or ends at a pole counts as a cable too.
  - Single circuit cables are on the layer `ECO_EMR_Circuito_aereo_simples` (exact, case-sensitive match). A cable on **any** other layer counts as double circuit.
  - A cable is linked to a pole only if its **first or last vertex** is within `0.5` drawing units of the pole insertion point. Intermediate vertices are ignored, so the cable polylines must be split at every pole.
- **Block attributes:** the drawing must contain blocks with the attribute tags `#-ESTRU.(PLANTA)`, `TIP.ESTRU.(PLANTA)`, `H/CAP.(PLANTA)`, `V.XXXX-XX(PLANTA)` and `V.XXXXX-XX(PLANTA)`, and at least one `INSERT`. They are dropped by name when the cables are selected, so a missing column raises a `KeyError`.

**Input prerequisites (`posts.xlsx`, on the Desktop):**
- One row per pole, extracted from the same drawing, with the columns `Position X`, `Position Y`, `Angle1`, `Angle2`, `H/CAP.(PLANTA)` and `V.XXXXX-XX(PLANTA)` (this last one is dropped, but it must exist).
- `H/CAP.(PLANTA)` starts with the 2-digit pole height in metres (`14/1000` → `14`).
- The pole coordinates must match the DWG, because they are compared with the cable ends.

**Running the script in AutoCAD:**
- The blocks `12-Simples` … `17-Duplo` (every height/circuit combination found) must already exist in the drawing.
- `-insert` is the command-line insert: the script sends the block name, the point, X scale `1`, Y scale `1` and the rotation. The blocks must have no attributes and must not have *Scale uniformly* on, or the values go into the wrong prompts.
- The poles are inserted on the current layer.

Extracting the AutoCAD data
=
- `dwg_folder` holds `all.dwg`, `temp_folder` is a working copy folder, and the DXF is written back to `dwg_folder`.
- The DWG is copied into `temp_folder` so the ODA File Converter only converts that one file.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) converts it to DXF (ACAD2018).
- `extract_data_from_dxf` reads every model space entity with `ezdxf` and returns a list of dicts. Nothing is loaded into DuckDB here.
- The DXF and the copied DWG are deleted at the end.

In [58]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np

dwg_folder = r"C:\Users\Usuario\Desktop\seriemas\RMT\all"
temp_folder = r"C:\Users\Usuario\Desktop\seriemas\RMT\all\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "all.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "all.dxf")

temp_dwg_file = os.path.join(temp_folder, "all.dwg")

data = extract_data_from_dxf(dxf_file)

print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


Entities as a DataFrame
=
Converts the extracted entities into `df`. The `INSERT` rows (poles, `*U` anonymous dynamic blocks) carry their attributes as columns, and the `LWPOLYLINE` rows carry their start/end points and length.

In [59]:
df = pd.DataFrame(data)
print(df)

           type                         layer handle name              X  \
0        INSERT        ECO_EMR_Postes_Parque6    825  *U2  458553.964795   
1        INSERT        ECO_EMR_Postes_Parque6    83F  *U3  458615.791768   
2        INSERT        ECO_EMR_Postes_Parque6    859  *U5  458660.899253   
3        INSERT        ECO_EMR_Postes_Parque6    873  *U6  458688.621283   
4        INSERT        ECO_EMR_Postes_Parque6    88D  *U4  458716.343314   
..          ...                           ...    ...  ...            ...   
949  LWPOLYLINE  ECO_EMR_Circuito_aereo_duplo  166F6  NaN            NaN   
950  LWPOLYLINE  ECO_EMR_Circuito_aereo_duplo  166F7  NaN            NaN   
951  LWPOLYLINE  ECO_EMR_Circuito_aereo_duplo  166F8  NaN            NaN   
952  LWPOLYLINE  ECO_EMR_Circuito_aereo_duplo  166F9  NaN            NaN   
953  LWPOLYLINE  ECO_EMR_Circuito_aereo_duplo  166FA  NaN            NaN   

                Y  rotation #-ESTRU.(PLANTA) TIP.ESTRU.(PLANTA)  \
0    7.814305e+06   

Quick check of the extracted data
=
`describe()`, `info()` and the count per entity `type`. In the last run there were 476 `INSERT` (poles) and 478 `LWPOLYLINE` (cable segments).

In [60]:
print(df.describe())
print(df.info())
print(df['type'].value_counts())

                   X             Y  rotation        start_x       start_y  \
count     476.000000  4.760000e+02     476.0     478.000000  4.780000e+02   
mean   457961.000386  7.814564e+06       0.0  457968.287500  7.814562e+06   
std      1378.606606  1.189571e+03       0.0    1376.060368  1.188714e+03   
min    454545.206813  7.812261e+06       0.0  454614.206718  7.812261e+06   
25%    457097.616703  7.813707e+06       0.0  457097.755130  7.813707e+06   
50%    458397.166666  7.814474e+06       0.0  458412.519308  7.814474e+06   
75%    458915.713003  7.815150e+06       0.0  458918.406112  7.815150e+06   
max    460858.634346  7.817170e+06       0.0  460858.634346  7.817170e+06   

               end_x         end_y      length  
count     478.000000  4.780000e+02  478.000000  
mean   457988.116938  7.814567e+06   69.664667  
std      1361.797295  1.173088e+03   12.084863  
min    454545.206813  7.812261e+06    0.010617  
25%    457098.234549  7.813707e+06   64.488076  
50%    45846

Loading the pole table
=
- Reads `posts.xlsx` (data extraction of the pole blocks) and drops `V.XXXXX-XX(PLANTA)`.
- Renames `Position X` / `Position Y` to `X` / `Y`.
- `angle` = `Angle1 + Angle2`, minus 360 when the sum is above 360. This is the insertion rotation used in the script. A sum of exactly 360 stays 360.

In [61]:
posts_df = pd.read_excel(r"C:\Users\Usuario\Desktop\posts.xlsx")
posts_df.drop(columns=['V.XXXXX-XX(PLANTA)'], inplace=True)
posts_df.rename(columns={'Position X': 'X', 'Position Y': 'Y'}, inplace=True)
posts_df['angle'] = np.where(
    posts_df['Angle1'] + posts_df['Angle2'] > 360,
    posts_df['Angle1'] + posts_df['Angle2'] - 360,
    posts_df['Angle1'] + posts_df['Angle2']
)
print(posts_df.describe())
print(posts_df.info())
print(posts_df)

           Angle2              X             Y      Angle1       angle
count  476.000000     476.000000  4.760000e+02  476.000000  476.000000
mean   249.924370  457961.000381  7.814564e+06   78.579832  237.747899
std     69.899586    1378.606606  1.189571e+03  108.828479   96.535805
min      0.000000  454545.206800  7.812261e+06    0.000000    0.000000
25%    270.000000  457097.616725  7.813707e+06    0.000000  190.000000
50%    270.000000  458397.166650  7.814474e+06   17.000000  270.000000
75%    270.000000  458915.712975  7.815150e+06  109.500000  285.000000
max    344.000000  460858.634300  7.817170e+06  357.000000  360.000000
<class 'pandas.DataFrame'>
RangeIndex: 476 entries, 0 to 475
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                476 non-null    str    
 1   #-ESTRU.(PLANTA)    476 non-null    str    
 2   Angle2              476 non-null    int64  
 3   H/CAP.(PLANTA)   

Selecting the cables
=
Keeps every `LWPOLYLINE` of the drawing as a cable and drops the columns that don't apply to cables. Only `layer` and the start/end coordinates are left.

There's no layer filter: any polyline in model space is treated as a cable.

In [62]:
cables_df = df[df['type'] == 'LWPOLYLINE'].copy()

cables_df.drop(columns=['type', 'name', 'handle', 'rotation', 'X', 'Y', '#-ESTRU.(PLANTA)', 'TIP.ESTRU.(PLANTA)', 'H/CAP.(PLANTA)', 'V.XXXX-XX(PLANTA)', 'V.XXXXX-XX(PLANTA)', 'length'], inplace=True)
print(cables_df.describe())
print(cables_df.info())
print(cables_df)

             start_x       start_y          end_x         end_y
count     478.000000  4.780000e+02     478.000000  4.780000e+02
mean   457968.287500  7.814562e+06  457988.116938  7.814567e+06
std      1376.060368  1.188714e+03    1361.797295  1.173088e+03
min    454614.206718  7.812261e+06  454545.206813  7.812261e+06
25%    457097.755130  7.813707e+06  457098.234549  7.813707e+06
50%    458412.519308  7.814474e+06  458469.274796  7.814479e+06
75%    458918.406112  7.815150e+06  458929.786496  7.815150e+06
max    460858.634346  7.817170e+06  460786.634344  7.817170e+06
<class 'pandas.DataFrame'>
Index: 478 entries, 23 to 953
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   layer    478 non-null    str    
 1   start_x  478 non-null    float64
 2   start_y  478 non-null    float64
 3   end_x    478 non-null    float64
 4   end_y    478 non-null    float64
dtypes: float64(4), str(1)
memory usage: 35.9 KB
None
               

Pole height
=
`Altura (m)` = first 2 characters of `H/CAP.(PLANTA)` (`14/1000` → `14`), kept as text because it becomes part of the block name.

In [63]:
posts_df['Altura (m)'] = posts_df['H/CAP.(PLANTA)'].astype(str).str[:2]
print(posts_df['Altura (m)'].value_counts())

Altura (m)
14    202
13    117
15     82
12     42
16     30
17      3
Name: count, dtype: int64


Finding the cables at each pole
=
- `distance_to_cable` returns the smaller of the distances from the pole to the cable's start point and to its end point. Despite the docstring, it doesn't measure the distance to the segment, so a cable that passes through a pole without a vertex there isn't found.
- For each pole, `nearby_cables` lists the layers of every cable with an end within `BUFFER_DISTANCE = 0.5` of the pole, and `cable_count` is its length.
- The loop compares every pole with every cable (`iterrows`), so it gets slow on large drawings.

In [64]:
def distance_to_cable(post_x, post_y, cable):
    """Calculate minimum distance from post to cable segment"""
    x1, y1, x2, y2 = cable['start_x'], cable['start_y'], cable['end_x'], cable['end_y']

    # Distance to start point
    d1 = np.sqrt((post_x - x1)**2 + (post_y - y1)**2)
    # Distance to end point
    d2 = np.sqrt((post_x - x2)**2 + (post_y - y2)**2)

    return min(d1, d2)

# For each post, find nearby cables
BUFFER_DISTANCE = 0.5

posts_df['nearby_cables'] = posts_df.apply(
    lambda row: [
        cable['layer']
        for _, cable in cables_df.iterrows()
        if distance_to_cable(row['X'], row['Y'], cable) <= BUFFER_DISTANCE
    ],
    axis=1
)

posts_df['cable_count'] = posts_df['nearby_cables'].apply(len)
print(posts_df)

                              Name #-ESTRU.(PLANTA)  Angle2 H/CAP.(PLANTA)  \
0    eco_ge_blocos_postes_vertices         P.01/05A     270        14/1000   
1    eco_ge_blocos_postes_vertices         P.02/03A       0        16/1500   
2    eco_ge_blocos_postes_vertices         P.02/06A     270        13/3000   
3    eco_ge_blocos_postes_vertices         P.02/10A     270        14/2500   
4    eco_ge_blocos_postes_vertices         P.02/13A     270        15/3000   
..                             ...              ...     ...            ...   
471  eco_ge_blocos_postes_vertices         P.02/06A     270        14/3000   
472  eco_ge_blocos_postes_vertices         P.02/07A     270        16/3000   
473  eco_ge_blocos_postes_vertices         P.02/09A     270        14/2000   
474  eco_ge_blocos_postes_vertices         P.00/08A     270        13/1500   
475  eco_ge_blocos_postes_vertices         P.00/14A     298        15/3000   

                          Layer TIP.ESTRU.(PLANTA) V.XXXX-XX(PL

Checking poles with more than 2 cables
=
- Most poles have 2 cables (one arriving, one leaving) and line ends have 1.
- Poles with more than 2 cables (branches, or duplicated polylines) are printed with a `CIRCLE X,Y 10` command, which can be pasted into AutoCAD to mark them for manual checking. These commands aren't written to a file.

In [65]:
posts_df['circle'] = 'CIRCLE ' + posts_df['X'].astype(str) + ',' + posts_df['Y'].astype(str) + ' 10'
print(posts_df['cable_count'].value_counts())
print(posts_df[posts_df['cable_count'] > 2][['X', 'Y', 'circle']])


cable_count
2    448
1     18
3      9
4      1
Name: count, dtype: int64
               X             Y                              circle
27   455424.9437  7.813720e+06  CIRCLE 455424.9437,7813719.9929 10
55   455307.2242  7.813252e+06   CIRCLE 455307.2242,7813251.879 10
148  458517.4441  7.813760e+06  CIRCLE 458517.4441,7813759.5344 10
219  456525.9771  7.813855e+06  CIRCLE 456525.9771,7813854.8648 10
222  456386.9915  7.814278e+06  CIRCLE 456386.9915,7814277.7685 10
225  457097.6354  7.814502e+06    CIRCLE 457097.6354,7814502.33 10
258  459859.1505  7.812746e+06  CIRCLE 459859.1505,7812745.9556 10
261  459526.8380  7.813230e+06   CIRCLE 459526.838,7813229.7554 10
463  459152.5089  7.816088e+06  CIRCLE 459152.5089,7816088.2025 10
465  459113.5593  7.815619e+06  CIRCLE 459113.5593,7815618.6061 10


Choosing single or double circuit
=
- A pole is `Simples` when all its cables are on `ECO_EMR_Circuito_aereo_simples`, otherwise it is `Duplo`.
- `condition1` (exactly 1 cable, on the simples layer) is already included in `condition2` (all cables on the simples layer), so it doesn't change the result.
- A pole with **no** cable nearby passes `condition2` (`all()` of an empty list is `True`) and becomes `Simples`. Check `cable_count == 0` before using the script.
- `Block name` = `<height>-Simples` or `<height>-Duplo`.

In [66]:
condition1 = (posts_df['cable_count'] == 1) & (posts_df['nearby_cables'].apply(lambda x: x[0]) == 'ECO_EMR_Circuito_aereo_simples')
condition2 = (posts_df['nearby_cables'].apply(lambda x: all(c == 'ECO_EMR_Circuito_aereo_simples' for c in x)))

is_simples = condition1 | condition2


posts_df['Block name'] = np.where(
    is_simples,
    posts_df['Altura (m)'].astype(str)+'-Simples',
    posts_df['Altura (m)'].astype(str)+'-Duplo'
)
print(posts_df)
print(posts_df['Block name'].value_counts())

                              Name #-ESTRU.(PLANTA)  Angle2 H/CAP.(PLANTA)  \
0    eco_ge_blocos_postes_vertices         P.01/05A     270        14/1000   
1    eco_ge_blocos_postes_vertices         P.02/03A       0        16/1500   
2    eco_ge_blocos_postes_vertices         P.02/06A     270        13/3000   
3    eco_ge_blocos_postes_vertices         P.02/10A     270        14/2500   
4    eco_ge_blocos_postes_vertices         P.02/13A     270        15/3000   
..                             ...              ...     ...            ...   
471  eco_ge_blocos_postes_vertices         P.02/06A     270        14/3000   
472  eco_ge_blocos_postes_vertices         P.02/07A     270        16/3000   
473  eco_ge_blocos_postes_vertices         P.02/09A     270        14/2000   
474  eco_ge_blocos_postes_vertices         P.00/08A     270        13/1500   
475  eco_ge_blocos_postes_vertices         P.00/14A     298        15/3000   

                          Layer TIP.ESTRU.(PLANTA) V.XXXX-XX(PL

Writing the AutoCAD script
=
Builds one command-line insert per pole:
`-insert <Block name> X,Y 1 1 <angle>`
(block, insertion point, X scale, Y scale, rotation).

Writes all the lines to **Postes.scr**. Run it in the target drawing with the `SCRIPT` command after the `NN-Simples` / `NN-Duplo` blocks have been defined there.

In [67]:
posts_df['Script'] = "-insert " + posts_df['Block name'].astype(str) + " " + \
                     posts_df['X'].astype(str) + "," +\
                     posts_df['Y'].astype(str) + \
                     " 1 1 " + posts_df['angle'].astype(str)
print(posts_df['Script'])

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\RMT\Postes.scr", 'w') as f:
    f.write('\n'.join(posts_df["Script"].astype(str)) + '\n')

0      -insert 14-Duplo 459799.9016,7812584.1083 1 1 360
1       -insert 16-Duplo 455783.3969,7814416.7013 1 1 66
2         -insert 13-Duplo 455887.019,7814469.82 1 1 261
3      -insert 14-Duplo 456140.5344,7814427.7251 1 1 281
4      -insert 15-Duplo 456332.2342,7814427.7251 1 1 248
                             ...                        
471     -insert 14-Duplo 458824.958,7814877.6697 1 1 181
472    -insert 16-Duplo 458847.2145,7814827.0128 1 1 226
473    -insert 14-Duplo 458916.0447,7814827.2933 1 1 315
474    -insert 13-Simples 458921.0395,7816554.6704 1 ...
475      -insert 15-Duplo 458553.9648,7814305.05 1 1 284
Name: Script, Length: 476, dtype: str
